In [ ]:
import pandas as pd
import numpy as np

from nltk.corpus import stopwords

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_DIR = "../data"

PREPS = ["normal", "stem", "lemma"]
VARIANTES = ["mx", "es", "cu"]

FEATURE_COLS = [
    "n_exc", "n_int", "n_may",
    "n_emo", "n_ris", "n_neg",
    "n_elo", "n_com", "n_pun"
]

STOP_WORDS = stopwords.words("spanish")

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [ ]:
def build_tfidf_prep():
    return ColumnTransformer([
        (
            "tfidf",
            TfidfVectorizer(
                stop_words=STOP_WORDS,
                max_features=10000,
                ngram_range=(1, 2)
            ),
            "MESSAGE_CLEAN"
        ),
        (
            "ling",
            "passthrough",
            FEATURE_COLS
        )
    ])

In [ ]:
def build_xgboost(scale_pos_weight):
    return XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.3,
        min_child_weight=1,

        scale_pos_weight=scale_pos_weight,

        objective="binary:logistic",
        eval_metric="logloss",
        booster="gbtree",
        tree_method="hist",

        random_state=RANDOM_STATE,
        n_jobs=-1
    )

In [ ]:
def calcular_scale_pos_weight(y):
    negativos = (y == 0).sum()
    positivos = (y == 1).sum()

    return negativos / positivos

In [ ]:
def evaluar_preprocessing(df_train):
    X = df_train[
        ["MESSAGE_CLEAN"] + FEATURE_COLS
    ]

    y = df_train["IS_IRONIC"].values

    spw = calcular_scale_pos_weight(y)

    pipeline = Pipeline([
        ("prep", build_tfidf_prep()),
        ("clf", build_xgboost(spw))
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=CV,
        scoring="f1_macro",
        n_jobs=1
    )

    return {
        "f1_macro_mean": scores.mean(),
        "f1_macro_std": scores.std(),
        "folds": scores,
        "scale_pos_weight": spw
    }

In [ ]:
resultados = []

for prep in PREPS:

    sufijo = "" if prep == "normal" else f"_{prep}"

    for variante in VARIANTES:

        path = f"{DATA_DIR}/train_clean{sufijo}_{variante}.csv"

        df_train = pd.read_csv(path)

        resultado = evaluar_preprocessing(df_train)

        resultados.append({
            "variante": variante,
            "preprocessing": prep,
            "f1_macro_mean": resultado["f1_macro_mean"],
            "f1_macro_std": resultado["f1_macro_std"],
            "scale_pos_weight": resultado["scale_pos_weight"],
            "folds": resultado["folds"]
        })

        print(
            f"{variante.upper()} | {prep:6} | "
            f"F1-Macro = {resultado['f1_macro_mean']:.4f} "
            f"± {resultado['f1_macro_std']:.4f}"
        )

In [ ]:
df_resultados = pd.DataFrame(resultados)

df_resultados[
    [
        "variante",
        "preprocessing",
        "f1_macro_mean",
        "f1_macro_std",
        "scale_pos_weight"
    ]
].sort_values(
    ["variante", "f1_macro_mean"],
    ascending=[True, False]
)

In [ ]:
ganadores = (
    df_resultados
    .sort_values("f1_macro_mean", ascending=False)
    .groupby("variante", as_index=False)
    .first()
)

ganadores[
    [
        "variante",
        "preprocessing",
        "f1_macro_mean",
        "f1_macro_std"
    ]
]